## 简单的RAG Demo

In [1]:
import langchain
langchain.__version__

'1.2.7'

In [23]:
# 最简RAG示例（不需要API key，用模拟数据）
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 准备文档（随便写一段文字）
with open("test.txt", "w") as f:
    f.write("PyTorch是一个深度学习框架。ONNX用于模型转换。TensorRT用于推理加速。")

# 2. 加载并分块
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

# 3. 向量化并存储
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(docs, embeddings)

# 4. 检索
query = "什么是TensorRT？"
results = vectorstore.similarity_search(query)
print(results[0].page_content)  # 应该输出关于TensorRT的那句话

PyTorch是一个深度学习框架。ONNX用于模型转换。TensorRT用于推理加速。


### 拓展：三句话拆分

In [27]:
# 1. 准备文档（随便写一段文字）
with open("test1.txt", "w") as f:
    f.write("PyTorch是一个深度学习框架。\nONNX用于模型转换。\nTensorRT用于推理加速。\n")

# 2. 加载并分块
loader = TextLoader("test1.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10, chunk_overlap=0, separators=['\n'])
docs = text_splitter.split_documents(documents)

print(f"分成了 {len(docs)} 个 chunk")  # 应该输出 3

# 3. 向量化并存储
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db_v2")

# 4. 检索
query = "什么是TensorRT？"
results = vectorstore.similarity_search(query)
print(results[0].page_content)  # 应该输出关于TensorRT的那句话

# 4. 检索
query = "什么是PyTorch？"
results = vectorstore.similarity_search(query)
print(results[0].page_content)  # 应该输出关于PyTorch的那句话

分成了 3 个 chunk
TensorRT用于推理加速。
PyTorch是一个深度学习框架。
